## Imports

In [2]:
from creating_dust_input_files import *
from creating_elliptic_wing import create_elliptic_wing
from running_DUST_files import *
import pandas as pd 
import numpy as np 
import os 
import pyvista as pv
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from velocity_in_front_of_kite import *

## Defining the paths

In [3]:
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

In [4]:
folder_path_DUST_files = os.path.join(parent_dir, "DUST files single") # path to folder with DUST files, here your dust input files will be created as well as the Output folder and Postprocessing folder
file_path_awebox = os.path.join(parent_dir, "awebox files/outputs_megawes_trajectory_vlm_results.csv") # path to the awebox file of the trajoctory
file_path_awebox_2 = os.path.join(parent_dir, "awebox files/outputs_megawes_trajectory_vlm3DOF_results.csv") # path to the awebox file of the trajoctory

In [5]:
n_rot = 1
res = 30
single_or_dual = "single"
particles_or_panels = "panels"
apply_pitch_correction = False
CL0 = 0
alpha_max = 0
alpha_min = 0

## Creating the Wing

In [6]:
AR = 12 # Aspect ratio of the wing
span = 42.47 # Span of the wing in meters
N_elements = 10 # Number of elements to discretize the wing
nelem_span = 4 # Number of subdivisions along the span for each element
end_chord = 1.0 # Chord length at the wing tips in meters
airfoil_profile = "NACA0012"
airfoil_table = "/Users/antoniamuhleck/Desktop/Hiwi/AWEWA/airfoils/naca0012.c81"

In [7]:
create_elliptic_wing(folder_path_DUST_files, AR, span, N_elements, nelem_span, airfoil_profile, end_chord, "ll", 3, airfoil_table )

## Creating the DUST files and running the simulation

In [ ]:
n_rot = 2  # Number of rotations to simulate (in case of dual kites, this is the number of half rotations, in case of single kites, this is the number of full rotations).
res = 50  # Resolution for the data. If res=1, all data is used; if res=2, every second data point is used...This is useful to reduce the time of the simulation.
single_or_dual = "single"
particles_or_panels = "panels" # "particles" for particle-based simulation, "panels" for panel-based simulation.
apply_pitch_correction = False # Whether to apply the pitch correction or not.
CL0 = 0 # Lift coefficient at zero angle of attack.(deduce from 3D drag curve)
alpha_max = 0 # Maximum angle of attack in degrees.(deduce from 3D lift curve)
alpha_min = 0 # Minimum angle of attack in degrees.(deduce from 3D lift curve)
box_length = 100 # Length of one side of the cubic simulation box (only used for particle-based simulation).
dof = 6 # either 6, if you did a "6 degrees of freedom" simulation in awebox or 3 for 3 degrees of freedom. 

In [14]:
create_dust_files(n_rot, res, file_path_awebox, folder_path_DUST_files, single_or_dual, apply_pitch_correction, CL0, alpha_max, alpha_min, particles_or_panels, box_length, dof)

In [16]:
# run_full_simulation(folder_path_DUST_files)

## Additional postprocessing

### Velocity in front of the wing

In [27]:
# velocities_20m, time = velocity_in_front_of_kite(n_rot, res, file_path_awebox, folder_path_DUST_files, single_or_dual, apply_pitch_correction, CL0, alpha_max, alpha_min, 20)

In [28]:
# plt.plot(time, np.array(velocities_20m[0])[:,0], label = "v_x")
# plt.plot(time, np.array(velocities_20m[0])[:,1], label = "v_y")
# plt.plot(time, np.array(velocities_20m[0])[:,2], label = "v_z")
# plt.legend()
# plt.title("d = 10m, first three rotations")
# plt.xlabel("time [s]")
# plt.ylabel("v[m/s]")
# plt.show()

### Heatmap in kite plane

The following code creates a heatmap in the plane in which the kites rotate. The function "calculate_flowfield_parameters_kite_plane" provides the parameters that can then be passed into the function "create_or_change_flow_field_analysis" as seen below. Then, you have to run the postprocessing again. 

In [29]:
t_flowfield_analysis = 70
min_xyz, max_xyz, v_normal = calculate_flowfield_parameters_kite_plane(file_path_awebox, "single", 500)
n_xyz = "(/40, 40, 40/)"
create_or_change_flow_field_analysis(folder_path_DUST_files, t_flowfield_analysis, t_flowfield_analysis, n_xyz, min_xyz, max_xyz, "heatmap_kite_plane" )


# run_dust_post(folder_path_DUST_files)

In the following, the heatmap in the kite plane is plotted. Here, the velocity that is shown the velocity component of the induced velocity that acts in the direction of the tether or orthogonal to the plane of the kites. The file that is outputted by DUST contains all velocity components including the freestream velocity, so you could also modify the following code to show the velocity component that you are interested in. 

In [30]:
file_path_hm_kite_plane = "/Users/antoniamuhleck/Desktop/Hiwi/AWEWA/DUST files single/Postprocessing/post_heatmap_kite_plane_0070.vtr"

In [31]:
plot_dict = pd.read_csv(file_path_awebox)
u_inf = np.average(plot_dict["outputs_aerodynamics_u_infty1_0"])

elevation_opt = np.arcsin(np.mean(plot_dict['x_q10_2']) / plot_dict['x_l_t_0'][0])
v_normal = [np.cos(elevation_opt), 0, np.sin(elevation_opt)]

In [17]:
mesh = pv.read(file_path_hm_kite_plane)

mesh["velocity"][:, 0] -= u_inf

# calculating velocity components orthogonal to the plane
v_components_orthogonal_to_plane = v_normal @ mesh["velocity"].T

# Convert the result back to a pyvista_ndarray
v_components_orthogonal_to_plane = np.clip(v_components_orthogonal_to_plane, -10, 10)
v_components_orthogonal_to_plane = pv.pyvista_ndarray(v_components_orthogonal_to_plane)

nx, ny, nz = mesh.dimensions
velocity_grid = v_components_orthogonal_to_plane.reshape((nz, ny, nx))

# works only for nz = nx
diagonal_velocity = np.array([velocity_grid[i, :, nz - i - 1] for i in range(nz)])
X = np.linspace(mesh.bounds[2], mesh.bounds[3], diagonal_velocity.shape[1])
Y = np.linspace(mesh.bounds[4], mesh.bounds[5], diagonal_velocity.shape[0])
X, Y = np.meshgrid(X, Y)
# Normalize the colormap to center at 0
norm = mcolors.TwoSlopeNorm(vmin=diagonal_velocity.min(), vcenter=0, vmax=diagonal_velocity.max())

# plt.figure(figsize=(8, 6))  # Adjust the figure size as needed
# ax = plt.gca()
# cset = plt.contourf(X, Y, diagonal_velocity, levels=30, cmap=plt.cm.RdBu_r, norm=norm)

# # Creating the plot
# # Adding a legend
# cbar = plt.colorbar(cset, label='$v_\mathrm{ind}$ [m/s]')
# cbar.set_label('$v_\mathrm{ind}$ [m/s]') 

# # Adding labels
# plt.xlabel('$e_\mathrm{1}$')
# plt.ylabel('$e_\mathrm{2}$')
# plt.show()

NameError: name 'file_path_hm_kite_plane' is not defined

### Heatmap in the xz plane

In this section, the induced velocity in the xz plane is plotted. The velocity component shown here is also the induced velocity that acts in the direction of the tethet or orthogonal to the plane of the kites. 

In [ ]:
t_flowfield_analysis = 170
min_xyz = calculate_flowfield_parameters_xz(file_path_awebox, t_flowfield_analysis, res,  single_or_dual) [0]
max_xyz = calculate_flowfield_parameters_xz(file_path_awebox, t_flowfield_analysis, res, single_or_dual) [1]
n_xyz = calculate_flowfield_parameters_xz(file_path_awebox, t_flowfield_analysis, res, single_or_dual) [2]
create_or_change_flow_field_analysis(folder_path_DUST_files, t_flowfield_analysis, t_flowfield_analysis, n_xyz, min_xyz, max_xyz, "heatmap_xz" )


# run_dust_post(folder_path_DUST_files)

In [ ]:
plot_dict = pd.read_csv(file_path_awebox)
u_inf = np.average(plot_dict["outputs_aerodynamics_u_infty1_0"])

elevation_opt = np.arcsin(np.mean(plot_dict['x_q10_2']) / plot_dict['x_l_t_0'][0])
v_normal = [np.cos(elevation_opt), 0, np.sin(elevation_opt)]

In [ ]:
file_path_hm_xz = os.path.join(parent_dir, "DUST files single/Postprocessing/post_heatmap_xz_0170.vtr")

In [ ]:
mesh = pv.read(file_path_hm_xz)

mesh["velocity"][:, 0] -= u_inf
v_components_orthogonal_to_plane = mesh["velocity"][:,0]
x, z = mesh.points[:, 0], mesh.points[:, 2]  

nx, ny = len(np.unique(x)), len(np.unique(z))
grid_values = v_components_orthogonal_to_plane.reshape((ny, nx)) 

grid_values = np.clip(grid_values, -12, 6)

norm = mcolors.TwoSlopeNorm(vmin=grid_values.min(), vcenter=0, vmax=grid_values.max())

X = np.linspace(mesh.bounds[0], mesh.bounds[1], grid_values.shape[1])
Y = np.linspace(mesh.bounds[4], mesh.bounds[5], grid_values.shape[0])
X, Y = np.meshgrid(X, Y)



# plt.figure(figsize=(10, 6))  
# ax = plt.gca()
# cset = plt.contourf(X, Y, grid_values, levels=30, cmap=plt.cm.RdBu_r, norm=norm)

# cbar = plt.colorbar(cset, label='$v_\mathrm{ind}$ [m/s]')
# cbar.set_label('$v_\mathrm{ind}$ [m/s]') 

# plt.xlabel('x')
# plt.ylabel('z')

# plt.show()
